In [1]:
import pandas as pd
import re

# 1. Cargar archivo
df = pd.read_excel(r"C:\Users\SD\Downloads\Normativa nacional víctimas 1869-2023.xlsx")

# 2. Ver columnas
print("Columnas:", df.columns)

# 3. Limpieza de texto
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    
    texto = str(texto).lower()
    
    # eliminar saltos de línea
    texto = texto.replace("\n", " ")
    
    # eliminar caracteres raros
    texto = re.sub(r"[^a-záéíóúñ0-9\s]", " ", texto)
    
    # eliminar espacios extra
    texto = re.sub(r"\s+", " ", texto).strip()
    
    return texto

# 4. Aplicar limpieza sobre la columna "Artículos"
df["texto_limpio"] = df["Artículos"].apply(limpiar_texto)

# 5. Crear dataset final
dataset = df[[
    "Tipo",
    "Número",
    "Título",
    "texto_limpio"
]].copy()

# 6. Eliminar filas sin texto
dataset = dataset[dataset["texto_limpio"] != ""]

# 7. Agregar longitud del texto (útil para análisis)
dataset["longitud_texto"] = dataset["texto_limpio"].apply(len)

# 8. Resetear índice
dataset = dataset.reset_index(drop=True)

# 9. Guardar dataset limpio

dataset.to_excel(r"C:\Users\SD\Downloads\dataset_normas.xlsx", index=False)

# 10. Info básica
print("\nDataset creado:")
print(dataset.shape)
print(dataset.head())

Columnas: Index(['Tipo', 'Número', 'Añosanción', 'texto completo', 'periodos',
       'periodos2', 'rango', 'presidencia', 'decadas', 'Título', 'Subtítulo',
       'Resumen', 'Artículos', 'caso_ok', 'caso', 'internacional_ok', 'tema',
       'subtema', 'tema2', 'subtema2', 'pol_ok1', 'pol_ok2', 'Comentarios'],
      dtype='object')

Dataset creado:
(144, 5)
  Tipo Número                              Título  \
0  Ley    340                        CODIGO CIVIL   
1  Ley   2372  CODIGO PROCESAL PENAL DE LA NACION   
2  Ley   2637                  CODIGO DE COMERCIO   
3  Ley   4960                       GASTO PéBLICO   
4  Ley   4964                  EMERGENCIA PéBLICA   

                                        texto_limpio  longitud_texto  
0  articulo 1079 la obligación de reparar el daño...             976  
1  articulo 170 la persona particularmente ofendi...            1477  
2  art 184 en caso de muerte o lesión de un viaje...             372  
3                   terremoto valpara

In [2]:
dataset

,Tipo,Número,Título,texto_limpio,longitud_texto
0,Ley,340,CODIGO CIVIL,articulo 1079 la obligación de reparar el daño...,976
1,Ley,2372,CODIGO PROCESAL PENAL DE LA NACION,articulo 170 la persona particularmente ofendi...,1477
2,Ley,2637,CODIGO DE COMERCIO,art 184 en caso de muerte o lesión de un viaje...,372
3,Ley,4960,GASTO PéBLICO,terremoto valparaíso agosto 1906,32
4,Ley,4964,EMERGENCIA PéBLICA,terremoto valparaíso agosto 1906,32
...,...,...,...,...,...
139,Ley,27547,LEY DE LA SOCIEDAD NACIONAL DE LA CRUZ ROJA,artículo 4 se autoriza a cruz roja argentina a...,1009
140,Decreto,459,DUELO NACIONAL,artículo 1 declárase duelo nacional en todo el...,226
141,Ley,27656,REPARACIÓN HISTÓRICA,artículo 1 dispónese la inscripción de la cond...,851
142,Ley,27696,PROGRAMA MÉDICO OBLIGATORIO DE LAS OBRAS SOCIA...,artículo 1 la presente ley tiene por objeto in...,1875


In [3]:
import re

def clasificar_derecho(texto):
    if not isinstance(texto, str): return 0
    texto = texto.lower()
    
    # Palabras que suelen indicar otorgamiento de derechos
    keywords = [
        r"otórgase", r"asistencia", r"indemnización", r"garantizar", 
        r"reparación", r"protección", r"beneficio", r"facultad"
    ]
    
    # Sujetos beneficiarios
    sujetos = [r"víctima", r"damnificado", r"afectado", r"particular"]

    # Si aparece una keyword cerca de un sujeto, es muy probable que sea Clase 1
    for k in keywords:
        for s in sujetos:
            # Busca si la keyword y el sujeto están a menos de 10 palabras de distancia
            if re.search(rf"{k}.*?{s}|{s}.*?{k}", texto):
                return 1
    return 0

In [4]:
dataset["label"] = dataset["texto_limpio"].apply(clasificar_derecho)

dataset["label"].value_counts()

label
0    82
1    62
Name: count, dtype: int64

In [5]:
dataset.sample(20)

,Tipo,Número,Título,texto_limpio,longitud_texto,label
65,Ley,25763,DERECHOS DEL NIÑO,artículo 8 1 los estados parte adoptarán medid...,2355,1
85,Decreto,851,PROGRAMA VERDAD Y JUSTICIA,artículo 1 apruébase la estructura organizativ...,730,1
124,Decreto,1338,ACUERDOS,artículo 1 apruébase el acuerdo de solución am...,576,0
88,Decreto,1343,ADMINISTRACION PUBLICA NACIONAL,secretaria de promocion y programas sanitarios...,468,0
107,Decreto,141,MINISTERIO DE JUSTICIA Y DERECHOS HUMANOS,art 2 sustitúyese del anexo ii del decreto n 1...,2782,1
10,Ley,13116,ACCIDENTE DE AVIACION,decreto 33 138 47,17,0
112,Ley,26795,CODIGO ADUANERO DEL MERCOSUR,capitulo viii envios de asistencia y salvament...,358,1
59,Decreto,258,MINISTERIO DEL INTERIOR,unidad ministro instituto nacional contra la d...,215,0
78,Decreto,1011,DUELO NACIONAL,artículo 1 declárase el 4 de agosto de 2006 fe...,217,0
15,Ley,16825,SUBSIDIO,no menciona la palabra víctimas habla de daños...,90,0


In [6]:
dataset.loc[10, "label"] = 1  # ejemplo

In [7]:
dataset = dataset.dropna(subset=["texto_limpio"])
dataset = dataset[dataset["texto_limpio"].str.len() > 30]

In [8]:
import json

ruta = r"C:\Users\SD\Downloads\dataset_normas.jsonl"

with open(ruta, "w", encoding="utf-8") as f:
    for _, row in dataset.iterrows():
        ejemplo = {
            "text": row["texto_limpio"],
            "label": int(row["label"])
        }
        json.dump(ejemplo, f, ensure_ascii=False)
        f.write("\n")

print("Dataset listo")

Dataset listo


In [10]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

# Cargar dataset
dataset = load_dataset("json", data_files=r"C:\Users\SD\Downloads\dataset_normas.jsonl")

# Dividir train/test
dataset = dataset["train"].train_test_split(test_size=0.2)

# Modelo (MULTILINGÜE → sirve para español)
model_name = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./modelo_normas",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

trainer.train()

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

C:\Users\SD\.anaconda\New folder\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SD\.cache\huggingface\hub\models--distilbert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
C:\Users\SD\.anaconda\New folder\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argumen

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=162, training_loss=0.6810443077558352, metrics={'train_runtime': 1019.8945, 'train_samples_per_second': 0.318, 'train_steps_per_second': 0.159, 'total_flos': 42919437164544.0, 'train_loss': 0.6810443077558352, 'epoch': 3.0})

In [11]:
predictions = trainer.predict(dataset["test"])

C:\Users\SD\.anaconda\New folder\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print("Precision:", precision_score(labels, preds))
print("Recall:", recall_score(labels, preds))
print("F1:", f1_score(labels, preds))

Accuracy: 0.75
Precision: 1.0
Recall: 0.5333333333333333
F1: 0.6956521739130436


In [14]:
trainer.save_model("./modelo_normas")
tokenizer.save_pretrained("./modelo_normas")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./modelo_normas\\tokenizer_config.json', './modelo_normas\\tokenizer.json')

In [15]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./modelo_normas")

texto = "La ley garantiza asistencia y reparación a víctimas del delito"

print(classifier(texto))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.9809122085571289}]
